# 01 - classes from week 18

Checks the dataset's stats files against the flow files, then picks the 10 most frequent services of week 18 and saves `config/classes.json` and `config/label_encoder.json` (the encoder was later switched to ranked order, see notebook 02).

Paths point to our shared Drive folder; change `RAW` and `ROOT` to run it elsewhere.

In [1]:
import time
from pathlib import Path
from remotezip import RemoteZip

URL = "https://zenodo.org/records/10608607/files/CESNET-TLS-Year22.zip?download=1"
RAW = Path(r"C:\cesnet_raw\WEEK-2022-18")
RAW.mkdir(parents=True, exist_ok=True)

for attempt in range(1, 4):
    try:
        with RemoteZip(URL, timeout=300, support_suffix_range=False) as z:
            stats = [i for i in z.infolist()
                     if "/WEEK-2022-18/" in i.filename and i.filename.endswith(".json")]
            for info in stats:
                dest = RAW / Path(info.filename).name
                if dest.exists() and dest.stat().st_size == info.file_size:
                    print("Already have:", dest.name)
                    continue
                dest.write_bytes(z.read(info))
                print("Saved:", dest.name, f"({info.file_size:,} bytes)")
        break
    except Exception as e:
        print(f"Attempt {attempt} failed: {e}")
        time.sleep(20)

print("\nStats files now present:")
for p in sorted(RAW.glob("stats-*.json")):
    print(" ", p.name)

Already have: stats-20220508.json
Already have: stats-20220502.json
Saved: stats-20220506.json (5,407 bytes)
Saved: stats-20220507.json (5,337 bytes)
Saved: stats-20220505.json (5,419 bytes)
Saved: stats-week.json (5,572 bytes)
Saved: stats-20220504.json (5,419 bytes)
Saved: stats-20220503.json (5,415 bytes)

Stats files now present:
  stats-20220502.json
  stats-20220503.json
  stats-20220504.json
  stats-20220505.json
  stats-20220506.json
  stats-20220507.json
  stats-20220508.json
  stats-week.json


In [2]:
import json
import pandas as pd

ROOT = Path(r"G:\My Drive\CESNET_Project")

def load_stats(path):
    d = json.loads(Path(path).read_text())
    return d["global"]["total-saved"], pd.Series(d["apps"], dtype="int64")

# Checks 1 and 2: stats file vs counting the real flow file
checks = {
    "2022-05-04": (RAW / "stats-20220504.json", ROOT / "data" / "flows-20220504.csv.xz"),
    "2022-05-08": (RAW / "stats-20220508.json", RAW / "flows-20220508.csv.xz"),
}
for day, (stats_path, flows_path) in checks.items():
    total, apps = load_stats(stats_path)
    real = pd.read_csv(flows_path, usecols=["APP"])["APP"].value_counts()
    both = pd.concat([apps.rename("stats"), real.rename("flows")], axis=1).fillna(0).astype(int)
    diff = (both["stats"] - both["flows"]).abs()
    print(f"{day}: stats total = {total:,} | rows in flow file = {int(real.sum()):,}")
    print(f"   services compared: {len(both)} | services that differ: {int((diff > 0).sum())} "
          f"| largest difference: {int(diff.max())}")

# Check 3: sum of the 7 daily files vs stats-week.json
daily = [load_stats(p) for p in sorted(RAW.glob("stats-2022*.json"))]
sum_total = sum(t for t, _ in daily)
sum_apps = pd.concat([a for _, a in daily], axis=1).fillna(0).sum(axis=1).astype("int64")
wk_total, wk_apps = load_stats(RAW / "stats-week.json")
wk_diff = (sum_apps.reindex(wk_apps.index, fill_value=0) - wk_apps).abs()
print(f"\nWeek 18: sum of daily totals = {sum_total:,} | stats-week total = {wk_total:,}")
print(f"   services that differ: {int((wk_diff > 0).sum())} | largest difference: {int(wk_diff.max())}")

2022-05-04: stats total = 2,617,956 | rows in flow file = 2,617,956
   services compared: 180 | services that differ: 0 | largest difference: 0
2022-05-08: stats total = 927,437 | rows in flow file = 927,437
   services compared: 180 | services that differ: 0 | largest difference: 0

Week 18: sum of daily totals = 14,561,533 | stats-week total = 14,561,533
   services that differ: 0 | largest difference: 0


In [3]:
import datetime, subprocess

# Exact CPU model for the paper's hardware description
cpu = subprocess.run(["powershell", "-NoProfile", "-Command",
                      "(Get-CimInstance Win32_Processor).Name"],
                     capture_output=True, text=True).stdout.strip()
with open(ROOT / "logs" / "machine_aya.txt", "a") as f:
    f.write(f"\nCPU model: {cpu}\n")
print("CPU model:", cpu)

# Flow counts per service per day (from the verified stats files)
per_day = {p.stem[6:]: load_stats(p)[1] for p in sorted(RAW.glob("stats-2022*.json"))}
counts = pd.DataFrame(per_day).fillna(0).astype("int64")
week_total = counts.sum(axis=1).sort_values(ascending=False)
total_flows = int(week_total.sum())

top10 = week_total.head(10)
print(f"\nWeek 18 flows: {total_flows:,} | services: {week_total.size}")
print("\nTop 12 by week-18 flow count (cut-off after 10):")
print(week_total.head(12).to_string())
gap = int(week_total.iloc[9] - week_total.iloc[10])
print(f"\nGap 10th vs 11th: {gap:,} flows ({gap / week_total.iloc[10]:.1%} of the 11th)")
share = top10.sum() / total_flows
print(f"Top 10 cover {share:.2%} of week-18 flows")

support = counts.loc[top10.index]
display(support)
print("Any zero-count day:", bool((support == 0).any().any()))

# Freeze classes and encoding
classes_cfg = {
    "source": "WEEK-2022-18 (2022-05-02 to 2022-05-08), daily stats files verified against flow files",
    "rule": "top 10 services by total flow count in training week 18 only; frozen for both chains",
    "classes_ranked": top10.index.tolist(),
    "week18_counts": {k: int(v) for k, v in top10.items()},
    "share_of_week18_flows": round(float(share), 4),
    "gap_10th_11th": gap,
    "created": str(datetime.date.today()),
}
(ROOT / "config" / "classes.json").write_text(json.dumps(classes_cfg, indent=2))

classes_sorted = sorted(top10.index.tolist())
enc = {"classes_sorted": classes_sorted,
       "index": {c: i for i, c in enumerate(classes_sorted)}}
(ROOT / "config" / "label_encoder.json").write_text(json.dumps(enc, indent=2))

support.to_csv(ROOT / "results" / "class_support_week18.csv")
pd.DataFrame({"week18_flows": week_total,
              "share": (week_total / total_flows).round(5)}).to_csv(
    ROOT / "results" / "service_counts_week18.csv")

with open(ROOT / "logs" / "run_log.md", "a") as f:
    f.write(f"- {datetime.date.today()} | Aya | A1: stats files verified (May 4, May 8, week total "
            f"exact match); top 10 classes from week 18 ({share:.2%} of flows, gap {gap:,}); "
            f"saved classes.json, label_encoder.json\n")

print("\nSaved: config/classes.json, config/label_encoder.json,")
print("       results/class_support_week18.csv, results/service_counts_week18.csv")
print("Encoding:", enc["index"])

CPU model: Intel(R) Core(TM) Ultra 5 115U

Week 18 flows: 14,561,533 | services: 180

Top 12 by week-18 flow count (cut-off after 10):
google-ads          857263
office365           479580
google-www          442811
youtube             434689
tiktok              407311
snapchat            401705
teams               393552
microsoft-update    383336
facebook-graph      378526
spotify             353426
outlook             331427
google-play         324250

Gap 10th vs 11th: 21,999 flows (6.6% of the 11th)
Top 10 cover 31.12% of week-18 flows


,20220502,20220503,20220504,20220505,20220506,20220507,20220508
google-ads,155678,171142,157573,167192,135309,34460,35909
office365,95800,95153,91286,87339,66277,21047,22678
google-www,83567,90269,81327,85273,65353,18563,18459
youtube,70611,88646,81397,84005,68280,21690,20060
tiktok,82053,73031,67137,68140,55263,28220,33467
snapchat,77857,79030,68305,91350,68945,6836,9382
teams,71920,74256,72294,68795,53315,25497,27475
microsoft-update,54257,68575,68207,66086,57028,35164,34019
facebook-graph,74479,78886,69517,68106,45191,18959,23388
spotify,63297,67713,63615,64820,48580,21574,23827


Any zero-count day: False

Saved: config/classes.json, config/label_encoder.json,
       results/class_support_week18.csv, results/service_counts_week18.csv
Encoding: {'facebook-graph': 0, 'google-ads': 1, 'google-www': 2, 'microsoft-update': 3, 'office365': 4, 'snapchat': 5, 'spotify': 6, 'teams': 7, 'tiktok': 8, 'youtube': 9}
